<a href="https://colab.research.google.com/github/Mahiman001/Data-Science-learning-practice/blob/main/1scipy_tests_cheat_sheet_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# STATISTICS HYPOTHESIS TESTING CHEAT SHEET
# SciPy | Compact DS-focused reference
# ============================================================

import numpy as np
from scipy import stats

alpha = 0.05

# ------------------------------------------------------------
# SHARED DATASET
# ------------------------------------------------------------

group_A = [32, 28, 31, 35, 29, 27, 33, 30]
group_B = [25, 24, 28, 23, 27, 26, 25, 29]
group_C = [38, 36, 35, 41, 39, 37, 34, 35]

# Paired data: BEFORE vs AFTER
before = [70, 65, 80, 75, 68, 72, 78, 74]
after  = [65, 62, 76, 70, 66, 68, 73, 70]

# Categorical data
gender_product = [
    [40, 20],
    [30, 30]
]

# Observed vs expected
observed = [18, 22, 17, 21, 19, 23]
expected = [20, 20, 20, 20, 20, 20]

# Correlation data
experience = [1,2,3,4,5,6,7,8,9,10]
job_satisfaction = [42,48,45,55,58,61,60,70,68,75]


# ============================================================
# 1. SHAPIRO-WILK TEST
# ============================================================

# WHY?
# Check whether data is approximately normally distributed.
# Useful before tests that assume normality.

stat, p = stats.shapiro(group_A)

print(stat, p)

# H0: Data is normally distributed
#
# p > 0.05 → Fail to reject H0 → normality assumption is okay
# p < 0.05 → Reject H0 → evidence of non-normality
#
# INSIGHT:
# Tells us whether a normality assumption is reasonable.


# ============================================================
# 2. LEVENE'S TEST
# ============================================================

# WHY?
# Check whether multiple groups have equal variances.
# Important assumption for regular ANOVA.

stat, p = stats.levene(group_A, group_B, group_C)

print(stat, p)

# H0: Group variances are equal
#
# p > 0.05 → Equal variance assumption is okay
# p < 0.05 → Variances are significantly different
#
# INSIGHT:
# p > 0.05 → Regular ANOVA can be used.
# p < 0.05 → Prefer Welch's ANOVA.


# ============================================================
# 3. ONE-WAY ANOVA
# ============================================================

# WHY?
# Compare means of 3+ INDEPENDENT groups.

F, p = stats.f_oneway(group_A, group_B, group_C)

print(F, p)

# H0: All group means are equal
#
# p < 0.05 → Reject H0
#            → At least ONE group mean differs
#
# p > 0.05 → Fail to reject H0
#            → No significant evidence of different means
#
# INSIGHT:
# ANOVA tells you THAT a difference exists,
# NOT exactly WHICH groups differ.
# Use Tukey HSD after significant ANOVA.


# ============================================================
# 4. WELCH'S ANOVA
# ============================================================

# WHY?
# Compare means of 3+ independent groups when
# variances are NOT equal.

F, p = stats.f_oneway(
    group_A, group_B, group_C,
    equal_var=False
)

print(F, p)

# H0: All group means are equal
#
# p < 0.05 → At least one mean differs
# p > 0.05 → No significant evidence of different means
#
# INSIGHT:
# Welch's ANOVA = ANOVA without equal-variance assumption.


# ============================================================
# 5. TWO-WAY ANOVA
# ============================================================

# WHY?
# Tests the effect of TWO categorical factors
# on one numerical outcome.
#
# Example:
# Delivery time = dependent variable
# Warehouse + Shift = factors
#
# Basic concept:
#
# Factor 1 → effect on outcome
# Factor 2 → effect on outcome
# Interaction → whether effect of one factor depends on another
#
# Usually performed with statsmodels, not scipy.
#
# Example:
#
# import statsmodels.api as sm
# from statsmodels.formula.api import ols
#
# model = ols(
#     'delivery_time ~ C(warehouse) + C(shift) + C(warehouse):C(shift)',
#     data=df
# ).fit()
#
# sm.stats.anova_lm(model, typ=2)
#
# p < 0.05 for a factor → that factor has a significant effect
# p < 0.05 for interaction → significant interaction effect


# ============================================================
# 6. REPEATED-MEASURES ANOVA
# ============================================================

# WHY?
# Compare 3+ means when the SAME subjects are measured
# repeatedly under different conditions/times.
#
# Example:
# Same people measured at:
# Week 1, Week 2, Week 3
#
# H0: All condition/time means are equal
#
# p < 0.05 → At least one condition/time differs
#
# INSIGHT:
# Used for DEPENDENT/RELATED groups, unlike one-way ANOVA.


# ============================================================
# 7. MANOVA
# ============================================================

# WHY?
# Compare groups when you have MULTIPLE dependent
# numerical variables simultaneously.
#
# Example:
# Warehouse → delivery_time + cost + customer_rating
#
# H0: Groups do not differ across the combined
# dependent variables.
#
# p < 0.05 → Significant multivariate group difference.
#
# INSIGHT:
# MANOVA = ANOVA with multiple dependent variables.


# ============================================================
# 8. TUKEY HSD
# ============================================================

# WHY?
# Run AFTER significant one-way ANOVA to find
# WHICH specific groups differ.

from statsmodels.stats.multicomp import pairwise_tukeyhsd

values = group_A + group_B + group_C
labels = (
    ['A'] * len(group_A) +
    ['B'] * len(group_B) +
    ['C'] * len(group_C)
)

result = pairwise_tukeyhsd(values, labels, alpha=0.05)
print(result)

# Look at:
# p-adj
#
# p-adj < 0.05 → That pair is significantly different
# p-adj > 0.05 → No significant difference between that pair
#
# INSIGHT:
# ANOVA → "something differs"
# Tukey → "A differs from B"


# ============================================================
# 9. INDEPENDENT T-TEST
# ============================================================

# WHY?
# Compare means of TWO independent groups.

t, p = stats.ttest_ind(group_A, group_B)

print(t, p)

# H0: Two population means are equal
#
# p < 0.05 → Significant difference in means
# p > 0.05 → No significant evidence of difference
#
# INSIGHT:
# 2 independent groups → t-test
# 3+ independent groups → ANOVA


# ============================================================
# 10. PAIRED T-TEST
# ============================================================

# WHY?
# Compare TWO measurements from the SAME subjects.
# Example: before vs after treatment.

t, p = stats.ttest_rel(before, after)

print(t, p)

# H0: Mean difference = 0
#
# p < 0.05 → Significant before/after difference
# p > 0.05 → No significant evidence of difference
#
# INSIGHT:
# Same subjects → paired t-test.


# ============================================================
# 11. MANN-WHITNEY U TEST
# ============================================================

# WHY?
# Compare TWO INDEPENDENT groups when normality is
# questionable / data is ordinal or non-normal.
#
# Non-parametric alternative to independent t-test.

U, p = stats.mannwhitneyu(
    group_A,
    group_B,
    alternative='two-sided'
)

print(U, p)

# H0: The two groups come from the same distribution
#     (often interpreted as no systematic difference)
#
# p < 0.05 → Significant difference
# p > 0.05 → No significant evidence of difference
#
# INSIGHT:
# 2 independent groups + non-normal/ordinal
# → Mann-Whitney U


# ============================================================
# 12. WILCOXON SIGNED-RANK TEST
# ============================================================

# WHY?
# Compare TWO RELATED/PAIRED measurements when the
# paired differences are not suitable for a paired t-test.
#
# Non-parametric alternative to paired t-test.

stat, p = stats.wilcoxon(before, after)

print(stat, p)

# H0: No systematic difference between paired measurements
#
# p < 0.05 → Significant difference
# p > 0.05 → No significant evidence of difference
#
# INSIGHT:
# Same subjects + non-normal/ordinal
# → Wilcoxon signed-rank


# ============================================================
# 13. CHI-SQUARE TEST OF INDEPENDENCE
# ============================================================

# WHY?
# Check whether TWO CATEGORICAL VARIABLES are associated.

chi2, p, dof, expected = stats.chi2_contingency(
    gender_product
)

print(chi2, p, dof, expected)

# H0: Variables are independent
#
# p < 0.05 → Significant association
# p > 0.05 → No significant evidence of association
#
# INSIGHT:
# Example: Is gender associated with product choice?


# ============================================================
# 14. CHI-SQUARE GOODNESS OF FIT
# ============================================================

# WHY?
# Compare OBSERVED frequencies with EXPECTED frequencies.

chi2, p = stats.chisquare(
    f_obs=observed,
    f_exp=expected
)

print(chi2, p)

# H0: Observed data follows the expected distribution
#
# p < 0.05 → Observed ≠ expected distribution
# p > 0.05 → Data is consistent with expected distribution
#
# INSIGHT:
# One categorical variable → observed vs expected.


# ============================================================
# 15. PEARSON CORRELATION
# ============================================================

# WHY?
# Measure strength and direction of a LINEAR relationship
# between two numerical variables.

r, p = stats.pearsonr(
    experience,
    job_satisfaction
)

print(r, p)

# H0: No linear correlation
#
# r → strength + direction
# p → statistical significance
#
# r > 0 → positive
# r < 0 → negative
# |r| close to 1 → strong
# |r| close to 0 → weak
#
# p < 0.05 → Significant linear relationship


# ============================================================
# 16. SPEARMAN CORRELATION
# ============================================================

# WHY?
# Measure strength/direction of a MONOTONIC relationship
# using ranks.
#
# Useful when relationship isn't necessarily linear or
# data doesn't meet Pearson assumptions.

rho, p = stats.spearmanr(
    experience,
    job_satisfaction
)

print(rho, p)

# H0: No monotonic relationship
#
# rho → strength + direction based on ranks
# p   → statistical significance
#
# rho > 0 → positive monotonic relationship
# rho < 0 → negative monotonic relationship
#
# p < 0.05 → Significant monotonic relationship


# ============================================================
# 17. UNIVERSAL p-VALUE RULE
# ============================================================

# p < 0.05
#     → Reject H0
#     → Evidence against H0
#
# p >= 0.05
#     → Fail to reject H0
#     → Not enough evidence against H0
#
# IMPORTANT:
# The RULE stays the same.
# What changes is WHAT H0 means for each test.


# ============================================================
# QUICK TEST SELECTION
# ============================================================

# NORMALITY CHECK
#       ↓
# Shapiro-Wilk
#
# 2 INDEPENDENT GROUPS
#       ↓
# Normal      → Independent t-test
# Non-normal  → Mann-Whitney U
#
# 2 RELATED/PAIRED GROUPS
#       ↓
# Normal differences     → Paired t-test
# Non-normal differences → Wilcoxon signed-rank
#
# 3+ INDEPENDENT GROUPS
#       ↓
# Equal variance     → One-way ANOVA
# Unequal variance   → Welch's ANOVA
#
# ANOVA significant
#       ↓
# Tukey HSD → Which groups differ?
#
# 2 CATEGORICAL VARIABLES
#       ↓
# Chi-square independence
#
# OBSERVED vs EXPECTED CATEGORIES
#       ↓
# Chi-square goodness of fit
#
# 2 NUMERICAL VARIABLES
#       ↓
# Linear relationship    → Pearson
# Monotonic/rank-based   → Spearman
#
# ============================================================
# THE BIG IDEA
# ============================================================
#
# 1. Identify the type of data
# 2. Identify independent vs paired groups
# 3. Check assumptions when required
# 4. Choose the test
# 5. Look mainly at p-value
# 6. Interpret H0 for THAT specific test
# 7. Report the actual statistic + p-value
#
# ============================================================